In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    ArrayType,
)

from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    size,
)

RAW_PATH = (
    "/Volumes/workspace/default/products/"
    "european_climate/raw/"
    "london_2024-01-01_2024-01-07.json"
)

BRONZE_TABLE = "bronze_weather_demo"

CITY = "London"
COUNTRY = "United Kingdom"
REGION = "Western Europe"
BATCH_ID = "London_2024-01-01_2024-01-07"
LOAD_TYPE = "TEST"
SOURCE = "Open-Meteo ERA5"

print("Raw file:", RAW_PATH)
print("Bronze table:", BRONZE_TABLE)


Raw file: /Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json
Bronze table: bronze_weather_demo


In [0]:
hourly_units_schema = StructType([
    StructField("time", StringType(), True),
    StructField("temperature_2m", StringType(), True),
    StructField("relative_humidity_2m", StringType(), True),
    StructField("apparent_temperature", StringType(), True),
    StructField("precipitation", StringType(), True),
    StructField("rain", StringType(), True),
    StructField("snowfall", StringType(), True),
    StructField("weather_code", StringType(), True),
    StructField("pressure_msl", StringType(), True),
    StructField("surface_pressure", StringType(), True),
    StructField("cloud_cover", StringType(), True),
    StructField("wind_speed_10m", StringType(), True),
    StructField("wind_direction_10m", StringType(), True),
])

hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),
    StructField("temperature_2m", ArrayType(DoubleType()), True),
    StructField("relative_humidity_2m", ArrayType(DoubleType()), True),
    StructField("apparent_temperature", ArrayType(DoubleType()), True),
    StructField("precipitation", ArrayType(DoubleType()), True),
    StructField("rain", ArrayType(DoubleType()), True),
    StructField("snowfall", ArrayType(DoubleType()), True),
    StructField("weather_code", ArrayType(IntegerType()), True),
    StructField("pressure_msl", ArrayType(DoubleType()), True),
    StructField("surface_pressure", ArrayType(DoubleType()), True),
    StructField("cloud_cover", ArrayType(DoubleType()), True),
    StructField("wind_speed_10m", ArrayType(DoubleType()), True),
    StructField("wind_direction_10m", ArrayType(DoubleType()), True),
])

raw_weather_schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("generationtime_ms", DoubleType(), True),
    StructField("utc_offset_seconds", IntegerType(), True),
    StructField("timezone", StringType(), True),
    StructField("timezone_abbreviation", StringType(), True),
    StructField("elevation", DoubleType(), True),
    StructField("hourly_units", hourly_units_schema, True),
    StructField("hourly", hourly_schema, True),
])

print("Explicit raw schema created.")


Explicit raw schema created.


In [0]:
raw_df = (
    spark.read
    .schema(raw_weather_schema)
    .option("multiLine", "true")
    .json(RAW_PATH)
)

assert raw_df.count() == 1, "Expected one raw API response per file"

hour_count = (
    raw_df
    .select(size(col("hourly.time")).alias("hour_count"))
    .first()["hour_count"]
)

print("Raw API records:", raw_df.count())
print("Hourly observations:", hour_count)

raw_df.printSchema()


Raw API records: 1
Hourly observations: 168
root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- utc_offset_seconds: integer (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- elevation: double (nullable = true)
 |-- hourly_units: struct (nullable = true)
 |    |-- time: string (nullable = true)
 |    |-- temperature_2m: string (nullable = true)
 |    |-- relative_humidity_2m: string (nullable = true)
 |    |-- apparent_temperature: string (nullable = true)
 |    |-- precipitation: string (nullable = true)
 |    |-- rain: string (nullable = true)
 |    |-- snowfall: string (nullable = true)
 |    |-- weather_code: string (nullable = true)
 |    |-- pressure_msl: string (nullable = true)
 |    |-- surface_pressure: string (nullable = true)
 |    |-- cloud_cover: string (nullable = true)
 |    |-- wind_speed_10m: string (nullable = true)


In [0]:

array_counts = (
    raw_df
    .select(
        size(col("hourly.time")).alias("time_count"),
        size(col("hourly.temperature_2m")).alias("temperature_count"),
        size(col("hourly.precipitation")).alias("precipitation_count"),
        size(col("hourly.wind_speed_10m")).alias("wind_count"),
    )
    .first()
)

counts = [
    array_counts["time_count"],
    array_counts["temperature_count"],
    array_counts["precipitation_count"],
    array_counts["wind_count"],
]

assert len(set(counts)) == 1, f"Hourly array-length mismatch: {counts}"

print("Hourly array lengths are consistent:", counts[0])


Hourly array lengths are consistent: 168


In [0]:
bronze_df = (
    raw_df
    .withColumn("city", lit(CITY))
    .withColumn("country", lit(COUNTRY))
    .withColumn("region", lit(REGION))
    .withColumn("source", lit(SOURCE))
    .withColumn("batch_id", lit(BATCH_ID))
    .withColumn("load_type", lit(LOAD_TYPE))
    .withColumn("raw_file_path", lit(RAW_PATH))
    .withColumn("load_timestamp", current_timestamp())
)

display(
    bronze_df.select(
        "city",
        "country",
        "region",
        "batch_id",
        "load_type",
        "source",
        "load_timestamp",
        "hourly",
    )
)


city country region batch_id load_type source load_timestamp hourly London United Kingdom Western Europe London_2024-01-01_2024-01-07 TEST Open-Meteo ERA5 2026-10-10T22:49:50.003Z List(List(2024-01-01T00:00, 2024-01-01T01:00, 2024-01-01T02:00, 2024-01-01T03:00, 2024-01-01T04:00, 2024-01-01T05:00, 2024-01-01T06:00, 2024-01-01T07:00, 2024-01-01T08:00, 2024-01-01T09:00, 2024-01-01T10:00, 2024-01-01T11:00, 2024-01-01T12:00, 2024-01-01T13:00, 2024-01-01T14:00, 2024-01-01T15:00, 2024-01-01T16:00, 2024-01-01T17:00, 2024-01-01T18:00, 2024-01-01T19:00, 2024-01-01T20:00, 2024-01-01T21:00, 2024-01-01T22:00, 2024-01-01T23:00, 2024-01-02T00:00, 2024-01-02T01:00, 2024-01-02T02:00, 2024-01-02T03:00, 2024-01-02T04:00, 2024-01-02T05:00, 2024-01-02T06:00, 2024-01-02T07:00, 2024-01-02T08:00, 2024-01-02T09:00, 2024-01-02T10:00, 2024-01-02T11:00, 2024-01-02T12:00, 2024-01-02T13:00, 2024-01-02T14:00, 2024-01-02T15:00, 2024-01-02T16:00, 2024-01-02T17:00, 2024-01-02T18:00, 2024-01-02T19:00, 2024-01-02T20:00, 2024-01-02T21:00, 2024-01-02T22:00, 2024-01-02T23:00, 2024-01-03T00:00, 2024-01-03T01:00, 2024-01-03T02:00, 2024-01-03T03:00, 2024-01-03T04:00, 2024-01-03T05:00, 2024-01-03T06:00, 2024-01-03T07:00, 2024-01-03T08:00, 2024-01-03T09:00, 2024-01-03T10:00, 2024-01-03T11:00, 2024-01-03T12:00, 2024-01-03T13:00, 2024-01-03T14:00, 2024-01-03T15:00, 2024-01-03T16:00, 2024-01-03T17:00, 2024-01-03T18:00, 2024-01-03T19:00, 2024-01-03T20:00, 2024-01-03T21:00, 2024-01-03T22:00, 2024-01-03T23:00, 2024-01-04T00:00, 2024-01-04T01:00, 2024-01-04T02:00, 2024-01-04T03:00, 2024-01-04T04:00, 2024-01-04T05:00, 2024-01-04T06:00, 2024-01-04T07:00, 2024-01-04T08:00, 2024-01-04T09:00, 2024-01-04T10:00, 2024-01-04T11:00, 2024-01-04T12:00, 2024-01-04T13:00, 2024-01-04T14:00, 2024-01-04T15:00, 2024-01-04T16:00, 2024-01-04T17:00, 2024-01-04T18:00, 2024-01-04T19:00, 2024-01-04T20:00, 2024-01-04T21:00, 2024-01-04T22:00, 2024-01-04T23:00, 2024-01-05T00:00, 2024-01-05T01:00, 2024-01-05T02:00, 2024-01-05T03:00, 2024-01-05T04:00, 2024-01-05T05:00, 2024-01-05T06:00, 2024-01-05T07:00, 2024-01-05T08:00, 2024-01-05T09:00, 2024-01-05T10:00, 2024-01-05T11:00, 2024-01-05T12:00, 2024-01-05T13:00, 2024-01-05T14:00, 2024-01-05T15:00, 2024-01-05T16:00, 2024-01-05T17:00, 2024-01-05T18:00, 2024-01-05T19:00, 2024-01-05T20:00, 2024-01-05T21:00, 2024-01-05T22:00, 2024-01-05T23:00, 2024-01-06T00:00, 2024-01-06T01:00, 2024-01-06T02:00, 2024-01-06T03:00, 2024-01-06T04:00, 2024-01-06T05:00, 2024-01-06T06:00, 2024-01-06T07:00, 2024-01-06T08:00, 2024-01-06T09:00, 2024-01-06T10:00, 2024-01-06T11:00, 2024-01-06T12:00, 2024-01-06T13:00, 2024-01-06T14:00, 2024-01-06T15:00, 2024-01-06T16:00, 2024-01-06T17:00, 2024-01-06T18:00, 2024-01-06T19:00, 2024-01-06T20:00, 2024-01-06T21:00, 2024-01-06T22:00, 2024-01-06T23:00, 2024-01-07T00:00, 2024-01-07T01:00, 2024-01-07T02:00, 2024-01-07T03:00, 2024-01-07T04:00, 2024-01-07T05:00, 2024-01-07T06:00, 2024-01-07T07:00, 2024-01-07T08:00, 2024-01-07T09:00, 2024-01-07T10:00, 2024-01-07T11:00, 2024-01-07T12:00, 2024-01-07T13:00, 2024-01-07T14:00, 2024-01-07T15:00, 2024-01-07T16:00, 2024-01-07T17:00, 2024-01-07T18:00, 2024-01-07T19:00, 2024-01-07T20:00, 2024-01-07T21:00, 2024-01-07T22:00, 2024-01-07T23:00), List(8.2, 7.9, 7.9, 7.9, 7.7, 7.5, 7.3, 7.2, 6.8, 6.7, 7.1, 7.9, 8.3, 8.5, 8.6, 8.6, 8.1, 8.0, 8.7, 9.3, 9.5, 9.7, 10.2, 10.9, 11.4, 12.1, 12.0, 11.6, 11.4, 11.4, 11.4, 11.0, 10.9, 11.0, 11.3, 11.7, 12.5, 12.7, 13.1, 12.6, 11.9, 10.8, 10.1, 9.9, 10.2, 10.4, 10.3, 10.1, 9.9, 9.8, 9.7, 9.4, 9.3, 9.6, 9.6, 9.4, 9.3, 9.4, 9.9, 10.5, 10.6, 10.7, 10.6, 10.5, 10.0, 9.8, 9.9, 9.5, 9.3, 8.8, 8.7, 8.2, 8.0, 8.1, 8.4, 8.2, 8.1, 7.6, 7.0, 6.5, 5.7, 5.7, 6.4, 7.4, 8.0, 8.1, 8.1, 8.0, 7.7, 7.8, 7.9, 7.9, 7.8, 7.8, 8.0, 7.1, 5.7, 5.0, 4.7, 5.1, 5.5, 5.6, 5.7, 5.7, 5.5, 5.4, 5.6, 6.1, 6.5, 6.8, 7.0, 7.0, 6.9, 6.8, 6.9, 6.8, 6.6, 6.3, 6.0, 5.7, 5.4, 5.7, 5.8, 5.7, 5.7, 5.6, 5.4, 5.4, 5.4, 5.5, 5.8, 6.2, 6.5, 6.5, 6.3, 6.2, 5.8, 5.4, 5.1, 4.7, 4.4, 4.2, 4.1, 4.2, 4.2, 3.9, 3.5, 3.1, 2.5, 2.2,

In [0]:

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

bronze_check = spark.table(BRONZE_TABLE)

print("Bronze records:", bronze_check.count())

bronze_check.select(
    "city",
    "batch_id",
    "load_type",
    "source",
    "load_timestamp",
).show(truncate=False)

print("Raw → Bronze completed successfully.")


Bronze records: 1
+------+----------------------------+---------+---------------+--------------------------+
|city  |batch_id                    |load_type|source         |load_timestamp            |
+------+----------------------------+---------+---------------+--------------------------+
|London|London_2024-01-01_2024-01-07|TEST     |Open-Meteo ERA5|2026-10-10 22:49:56.000339|
+------+----------------------------+---------+---------------+--------------------------+

Raw → Bronze completed successfully.
